In [6]:
from abc import ABC, abstractmethod
from datetime import datetime
import json


# ABSTRACT CLASS

class Kisi(ABC):
    def __init__(self, id, ad):
        self.__id = id
        self.__ad = ad

    def get_id(self):
        return self.__id

    def get_ad(self):
        return self.__ad

    @abstractmethod
    def bilgileri_goster(self):
        pass



# UYE

class Uye(Kisi):
    def __init__(self, id, ad):
        super().__init__(id, ad)

    def bilgileri_goster(self):
        return f"Üye ID: {self.get_id()} | Ad: {self.get_ad()}"



# KUTUPHANECI

class Kutuphaneci(Kisi):
    def __init__(self, id, ad):
        super().__init__(id, ad)

    def bilgileri_goster(self):
        return f"Kütüphaneci ID: {self.get_id()} | Ad: {self.get_ad()}"



# KITAP

class Kitap:
    def __init__(self, kitap_id, ad, yazar, stok):
        self.__kitap_id = kitap_id
        self.__ad = ad
        self.__yazar = yazar
        self.__stok = stok

    def get_id(self):
        return self.__kitap_id

    def get_ad(self):
        return self.__ad

    def get_yazar(self):
        return self.__yazar

    def get_stok(self):
        return self.__stok

    def stok_azalt(self):
        if self.__stok > 0:
            self.__stok -= 1

    def stok_arttir(self):
        self.__stok += 1

    def bilgileri_goster(self):
        return f"ID:{self.__kitap_id} | {self.__ad} | {self.__yazar} | Stok:{self.__stok}"



# ODUNC ISLEMI
#Gecikme cezasını veriliş tarihinden 10 gün sonra iade edilmeyen her gün için 3tl para cezası eklenir

def gecikme_cezasi(self):

    verilis = datetime.strptime(
        self.verilis_tarihi,
        "%Y-%m-%d"
    )

    bugun = datetime.now()

    gun_farki = (bugun - verilis).days

    if gun_farki <= 10:
        return 0

    return (gun_farki - 10) * 3


# KUTUPHANE SISTEMI

class KutuphaneSistemi:

    def __init__(self):
        self.kitaplar = []
        self.uyeler = []
        self.odunc_listesi = []


    # KITAP EKLE

    def kitap_ekle(self):

        try:
            kitap_id = input("Kitap ID: ")
            ad = input("Kitap Adı: ")
            yazar = input("Yazar: ")
            stok = int(input("Stok: "))

            kitap = Kitap(kitap_id, ad, yazar, stok)
            self.kitaplar.append(kitap)

            print("Kitap eklendi.")

        except:
            print("Hatalı giriş.")


    # UYE EKLE

    def uye_ekle(self):

        try:
            uye_id = input("Üye ID: ")
            ad = input("Üye Adı: ")

            uye = Uye(uye_id, ad)
            self.uyeler.append(uye)

            print("Üye eklendi.")

        except:
            print("Hatalı giriş.")


    # KITAPLARI LISTELE

    def kitaplari_listele(self):

        if not self.kitaplar:
            print("Kitap bulunamadı.")
            return

        for kitap in self.kitaplar:
            print(kitap.bilgileri_goster())


    # UYELERI LISTELE

    def uyeleri_listele(self):

        if not self.uyeler:
            print("Üye bulunamadı.")
            return

        for uye in self.uyeler:
            print(uye.bilgileri_goster())


    # KITAP ARA

    def kitap_ara(self):

        aranacak = input("Kitap adı giriniz: ").lower()

        bulundu = False

        for kitap in self.kitaplar:
            if aranacak in kitap.get_ad().lower():
                print(kitap.bilgileri_goster())
                bulundu = True

        if not bulundu:
            print("Kitap bulunamadı.")


    # ODUNC VER

    def kitap_odunc_ver(self):

        uye_id = input("Üye ID: ")
        kitap_id = input("Kitap ID: ")

        for kitap in self.kitaplar:

            if kitap.get_id() == kitap_id:

                if kitap.get_stok() == 0:
                    print("Stokta kitap yok.")
                    return

                verilis_tarihi = input(
                    "Veriliş Tarihi (YYYY-AA-GG): "
                )

                kitap.stok_azalt()

                odunc = OduncIslemi(
                    uye_id,
                    kitap_id,
                    verilis_tarihi
                )

                self.odunc_listesi.append(odunc)

                print("Kitap ödünç verildi.")
                return

        print("Kitap bulunamadı.")


    # IADE AL

    def kitap_iade_al(self):

        kitap_id = input("İade edilen kitap ID: ")

        for odunc in self.odunc_listesi:

            if odunc.kitap_id == kitap_id and odunc.iade_tarihi is None:

                odunc.kitap_iade()

                for kitap in self.kitaplar:
                    if kitap.get_id() == kitap_id:
                        kitap.stok_arttir()

                print("Kitap iade edildi.")
                return

        print("Kayıt bulunamadı.")


    # CEZA HESAPLA

    def gecikme_cezasi_hesapla(self):

        kitap_id = input("Kitap ID: ")

        for odunc in self.odunc_listesi:

            if odunc.kitap_id == kitap_id:

                ceza = odunc.gecikme_cezasi()

                print(f"Gecikme cezası: {ceza} TL")
                return

        print("Kayıt bulunamadı.")


    # VERILERI KAYDET

    def verileri_kaydet(self):

        veri = {
            "kitaplar": [
                {
                    "id": k.get_id(),
                    "ad": k.get_ad(),
                    "yazar": k.get_yazar(),
                    "stok": k.get_stok()
                }
                for k in self.kitaplar
            ],
            "uyeler": [
                {
                    "id": u.get_id(),
                    "ad": u.get_ad()
                }
                for u in self.uyeler
            ]
        }

        with open("kutuphane.json", "w", encoding="utf-8") as dosya:
            json.dump(veri, dosya, ensure_ascii=False, indent=4)

        print("Veriler kaydedildi.")


    # VERILERI YUKLE

    def verileri_yukle(self):

        try:

            with open("kutuphane.json", "r", encoding="utf-8") as dosya:

                veri = json.load(dosya)

            self.kitaplar.clear()
            self.uyeler.clear()

            for k in veri["kitaplar"]:
                self.kitaplar.append(
                    Kitap(
                        k["id"],
                        k["ad"],
                        k["yazar"],
                        k["stok"]
                    )
                )

            for u in veri["uyeler"]:
                self.uyeler.append(
                    Uye(
                        u["id"],
                        u["ad"]
                    )
                )

            print("Veriler yüklendi.")

        except FileNotFoundError:
            print("Kayıtlı dosya bulunamadı.")



# MENU

def menu():

    sistem = KutuphaneSistemi()

    while True:

        print("\nKÜTÜPHANE OTOMASYON SİSTEMİ")
        print("1- Kitap Ekle")
        print("2- Üye Ekle")
        print("3- Kitapları Listele")
        print("4- Üyeleri Listele")
        print("5- Kitap Ödünç Ver")
        print("6- Kitap İade Al")
        print("7- Gecikme Cezası Hesapla")
        print("8- Kitap Ara")
        print("9- Verileri Kaydet")
        print("10- Verileri Yükle")
        print("0- Çıkış")

        secim = input("Seçiminiz: ")

        if secim == "1":
            sistem.kitap_ekle()

        elif secim == "2":
            sistem.uye_ekle()

        elif secim == "3":
            sistem.kitaplari_listele()

        elif secim == "4":
            sistem.uyeleri_listele()

        elif secim == "5":
            sistem.kitap_odunc_ver()

        elif secim == "6":
            sistem.kitap_iade_al()

        elif secim == "7":
            sistem.gecikme_cezasi_hesapla()

        elif secim == "8":
            sistem.kitap_ara()

        elif secim == "9":
            sistem.verileri_kaydet()

        elif secim == "10":
            sistem.verileri_yukle()

        elif secim == "0":
            print("Program kapatıldı.")
            break

        else:
            print("Geçersiz seçim.")



# PROGRAM BASLAT

if __name__ == "__main__":
    menu()


KÜTÜPHANE OTOMASYON SİSTEMİ
1- Kitap Ekle
2- Üye Ekle
3- Kitapları Listele
4- Üyeleri Listele
5- Kitap Ödünç Ver
6- Kitap İade Al
7- Gecikme Cezası Hesapla
8- Kitap Ara
9- Verileri Kaydet
10- Verileri Yükle
0- Çıkış


KeyboardInterrupt: Interrupted by user